# Real-data validation: FBP3D TomoBAR on I13 dataset 3

Reconstruct `i13_dataset3.npz` and compare numerical checks with the saved slice,
region of interest and line profile.

## Data and execution

Use `i13_dataset3.npz` from [Zenodo record 17252190](https://zenodo.org/records/17252190).
Run from the repository root:

```bash
python .scripts/download_zenodo.py Zenodo
python -m pip install -e '.[validation]'
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/reconstruction/direct/fbp3d_i13_dataset3.ipynb
```

Requires an NVIDIA GPU with compatible CuPy, CUDA and ToMoBAR (and ASTRA for
ASTRA-based methods). Run GPU notebooks serially. Set `HTTOMOLIBGPU_DATA_DIR`
for an archive stored elsewhere. Documentation renders saved outputs;
use `--overwrite` to refresh them.


In [ ]:
import gc
import math
import pytest
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from numpy.testing import assert_allclose

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log
from httomolibgpu.recon.algorithm import FBP3d_tomobar

# Locate the checkout from either the repository root or the notebook directory.
repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
data_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if data_dir is None:
    if repo_root is None:
        raise FileNotFoundError(
            "Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory"
        )
    data_dir = repo_root / "Zenodo"
data_path = Path(data_dir).expanduser() / "i13_dataset3.npz"
if not data_path.is_file():
    raise FileNotFoundError(
        f"Missing {data_path}; run .scripts/download_zenodo.py first"
    )

checksum = hashlib.sha256()
with data_path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        checksum.update(chunk)
props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
provenance = {
    "dataset": data_path.name,
    "zenodo_record": "17252190",
    "sha256": checksum.hexdigest(),
    "python": platform.python_version(),
    "packages": {
        name: version(name)
        for name in ("httomolibgpu", "tomobar", "astra-toolbox", "numpy")
    },
    "cupy": cp.__version__,
    "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
    "cuda_driver": cp.cuda.runtime.driverGetVersion(),
    "gpu": (
        props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    ),
}
print(json.dumps(provenance, indent=2))
from httomolibgpu.misc.morph import sino_360_to_180

## Preprocessing

Input axes are `(angle, detector row, detector column)`.
Apply default dark/flat correction and `minus_log`. Stitch all three detector
rows with overlap `438.173828` and `side="left"`, then use the first
3000 angles in radians.


In [ ]:
with np.load(data_path) as archive:
    projdata = cp.asarray(archive["projdata"])
    flats = cp.asarray(archive["flats"])
    darks = cp.asarray(archive["darks"])
    angles = archive["angles"].copy()
data_normalised = minus_log(dark_flat_field_correction(projdata, flats, darks))
del projdata, flats, darks
gc.collect()
cp.get_default_memory_pool().free_all_blocks()
assert bool(cp.isfinite(data_normalised).all())
print("Corrected input:", data_normalised.shape)
stitched_data = sino_360_to_180(data_normalised, overlap=438.173828, side="left")
del data_normalised
gc.collect()
cp.get_default_memory_pool().free_all_blocks()
recon_angles = np.deg2rad(angles[:3000])
assert stitched_data.shape[0] == len(recon_angles)
assert bool(cp.isfinite(stitched_data).all())
print("Stitched projections:", stitched_data.shape)
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
im = ax.imshow(
    stitched_data[:, stitched_data.shape[1] // 2, :].get(), cmap="gray", aspect="auto"
)
ax.set(
    title="Stitched central-row sinogram",
    xlabel="Detector column",
    ylabel="Angle index",
)
fig.colorbar(im, ax=ax, label="Negative log transmission")
plt.show()
plt.close(fig)

## Reconstruction and regression assertions

Use centre 2560 pixels, no detector padding and SINC filter cutoff 0.35.
Compare mean, minimum and maximum with the recorded reference values.
Output axes are `(Y, detector row, X)`. Numerical assertions run after plotting.


In [ ]:
parameters = {"center": 2560, "detector_pad": 0, "filter_freq_cutoff": 0.35}
print("Reconstruction parameters:", parameters)
recon_gpu = FBP3d_tomobar(stitched_data, recon_angles, **parameters)
assert recon_gpu.flags.c_contiguous
assert recon_gpu.dtype == cp.float32
assert recon_gpu.shape == (5120, 3, 5120)
assert bool(cp.isfinite(recon_gpu).all())
recon_data = cp.asnumpy(recon_gpu)
del recon_gpu, stitched_data
gc.collect()
cp.get_default_memory_pool().free_all_blocks()
observed_mean = float(recon_data.mean(dtype=np.float64))
observed_max = float(recon_data.max())
observed_min = float(recon_data.min())
print(
    json.dumps(
        {
            "shape": list(recon_data.shape),
            "dtype": str(recon_data.dtype),
            "mean": observed_mean,
            "minimum": observed_min,
            "maximum": observed_max,
            "sum_float64": float(recon_data.sum(dtype=np.float64)),
        },
        indent=2,
    )
)
print("All structural validation checks passed.")

In [ ]:
image = recon_data[:, recon_data.shape[1] // 2, :]
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
limits = {"vmin": -0.001, "vmax": 0.0015, "cmap": "gray"}
im = axes[0].imshow(image, **limits)
axes[0].set(title="Central reconstructed slice", xlabel="X pixel", ylabel="Y pixel")
axes[1].imshow(image[1800:2800, 2000:3000], extent=(2000, 3000, 2800, 1800), **limits)
axes[1].set(title="Fixed region of interest", xlabel="X pixel", ylabel="Y pixel")
fig.colorbar(im, ax=axes, label="Reconstructed value (library units)")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
ax.plot(image[image.shape[0] // 2, :])
ax.set(
    title="Central horizontal profile", xlabel="X pixel", ylabel="Reconstructed value"
)
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)

In [ ]:
# Reference: saved successful run, all three rows, shape (5120, 3, 5120).
expected_mean = 7.996705091643615e-05
expected_max = 0.0016445642104372382
expected_min = -0.0011296591255813837
assert_allclose(
    observed_mean, expected_mean, rtol=1e-3, atol=1e-8, err_msg="Mean changed"
)
assert_allclose(
    observed_max, expected_max, rtol=1e-3, atol=1e-8, err_msg="Maximum changed"
)
assert_allclose(
    observed_min, expected_min, rtol=1e-3, atol=1e-8, err_msg="Minimum changed"
)
print("Mean, maximum and minimum regression checks passed.")

del recon_data, image
gc.collect()
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()